# Execução do projeto TrashNet no Google Colab

Este notebook usa o projeto versionado no GitHub e mantém dados, modelos e históricos no Google Drive para que a equipe consiga retomar a execução.

**Antes de executar:** crie o repositório público no GitHub seguindo `docs/PASSO_A_PASSO_COLAB_VSCODE_GITHUB.md` e substitua o endereço de exemplo na célula abaixo.

No menu do Colab, escolha **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU**. A GPU concedida depende da disponibilidade e da cota da conta.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')

# TROQUE pelo endereço HTTPS público do repositório criado pela equipe.
REPO_URL = 'https://github.com/felipecarvalho98-bit/trashnet-mlp-cnn-unifor.git'
PROJECT_DIR = Path('/content/drive/MyDrive/IA_UNIFOR/trashnet-mlp-cnn-unifor')
PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)

if 'SEU_USUARIO' in REPO_URL:
    raise ValueError('Edite REPO_URL com o endereço real do GitHub antes de continuar.')

if not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR, check=True)

# A leitura repetida das imagens acontece no disco local da VM do Colab,
# evitando milhares de acessos ao Google Drive durante cada época.
LOCAL_DATA_DIR = Path('/content/trashnet_data')
os.environ['TRASHNET_DATA_DIR'] = str(LOCAL_DATA_DIR)
os.environ['TRASHNET_OUTPUT_DIR'] = str(PROJECT_DIR / 'outputs')

os.chdir(PROJECT_DIR)
print('Pasta do projeto:', PROJECT_DIR)
print('Dados temporários:', LOCAL_DATA_DIR)
print('Resultados persistentes:', PROJECT_DIR / 'outputs')


**Armazenamento:** o código e `outputs/` ficam na pasta do projeto no Google Drive. O ZIP e as imagens são baixados para `/content/trashnet_data`, no disco local da VM, reduzindo o custo de leitura repetitiva durante o treinamento. Se o runtime for recriado, os dados são baixados/extraídos novamente; modelos, históricos e checkpoints permanecem no Drive.


## 1. Instalar dependências

O Colab normalmente já traz TensorFlow no ambiente. Instalamos as demais bibliotecas do projeto sem forçar a troca da versão do TensorFlow fornecida pelo runtime.

In [ ]:
%pip install -q numpy scikit-learn Pillow matplotlib psutil pandas


## 2. Confirmar GPU e ambiente

Não assuma que a GPU está sendo usada apenas porque a opção foi selecionada. Confira se o TensorFlow detectou o dispositivo e se a multiplicação de matrizes foi executada em `/device:GPU:0`. O `nvidia-smi` registra o modelo e a memória total da placa. Durante os treinos, o código mede memória amostrada e tempos por época; veja `docs/ANALISE_GPU.md` para as limitações dessas medições.


In [ ]:
import tensorflow as tf
import platform

gpus = tf.config.list_physical_devices('GPU')
print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print('Detalhes:', tf.config.experimental.get_device_details(gpu))
    except Exception as exc:
        print('Não foi possível configurar memória dinâmica:', exc)

if not gpus:
    print('ATENÇÃO: este ambiente não detectou GPU. Verifique Ambiente de execução > Alterar o tipo > GPU.')
# Teste real de operação no dispositivo, além da detecção pelo TensorFlow.
if gpus:
    with tf.device('/GPU:0'):
        teste_gpu = tf.linalg.matmul(tf.random.normal((1024, 1024)), tf.random.normal((1024, 1024)))
    print('Operação de teste executada em:', teste_gpu.device)
else:
    print('A operação será executada em CPU; selecione GPU nas configurações do Colab e reinicie o ambiente.')


## 3. Análise exploratória e divisão dos dados

O código baixa o ZIP de referência, valida o SHA-256, percorre todas as imagens, registra dimensão/cor/integridade e gera uma divisão estratificada 70/15/15 com semente 42. A execução pode levar alguns minutos.

In [ ]:
!python -m src.prepare_data

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

inspection = json.loads(Path('outputs/metricas/inspecao_dataset.json').read_text(encoding='utf-8'))
print('Total validado:', inspection['total_valid_images'])
print('Contagem por classe:', inspection['counts_by_class'])
print('Dimensões:', inspection['dimensions_width_x_height'])
print('Modos de cor:', inspection['original_pil_modes'])
print('Imagens corrompidas:', len(inspection['corrupted_files']))
print('Grupos duplicados:', len(inspection['exact_pixel_duplicates']))

splits = json.loads(Path('outputs/splits.json').read_text(encoding='utf-8'))
display(pd.DataFrame(splits['counts_by_split_and_class']).T)
for name in ['fig1_distribuicao.png','fig2_exemplos.png','fig3_resolucoes.png','fig4_pesos_classe.png']:
    display(Image(filename=f'outputs/figuras/{name}'))

## 4. Testar um experimento

Execute primeiro E3 (CNN 128 × 96 sem pesos) para verificar se o treinamento inicia e se a GPU tem memória suficiente. Esse comando faz um treinamento completo com early stopping, não é apenas um teste de uma época.

In [ ]:
!python -m src.train --exp E3

## 5. Treinar os seis experimentos

O comando `--skip-existing` preserva os modelos já concluídos. Se a sessão for interrompida, execute esta célula novamente na mesma pasta do Drive. O script tentará lote 32, depois 16 e 8 se ocorrer falta de memória, registrando a configuração realmente usada.

In [ ]:
!python -m src.train --all --skip-existing

## 6. Avaliar os modelos e gerar resultados

A avaliação calcula o F1 macro de validação para selecionar o modelo de referência e, depois da seleção, avalia os seis modelos no conjunto de teste. Por padrão, a inferência usa o lote registrado no treinamento de cada modelo, evitando falta de memória desnecessária. Não use o teste para ajustar os modelos.

In [ ]:
!python -m src.evaluate

In [ ]:
from IPython.display import display, Image

for filename in ['tabela5_custo.csv','tabela6_desempenho.csv','tabela7_por_classe.csv']:
    print(filename)
    display(pd.read_csv(f'outputs/metricas/{filename}'))

for name in ['fig5_curvas_aprendizagem.png','fig6_matrizes_confusao.png','fig7_resolucao_e_custo.png','fig8_erros_modelo_referencia.png']:
    display(Image(filename=f'outputs/figuras/{name}'))

## 7. Conferência antes de fechar o Colab

Confirme que existem seis arquivos `.keras`, seis históricos, seis configurações, três tabelas CSV e as Figuras 1–8. Confira também os arquivos `config_E*.json`: hardware, lote, épocas, tempo, RAM e campos de GPU. Campos `null` significam que o driver/API não forneceu aquela métrica; não estime esses valores.

**Importante:** os modelos `.keras` e checkpoints não são enviados ao GitHub; ficam no Google Drive e devem ser compartilhados com o professor conforme a orientação da disciplina. Versione código, notebook, tabelas, métricas e figuras pequenas. Atualize o artigo com os valores reais e revise o texto em grupo.
